# Deteccion de Fraude usando ML

In [ ]:
!pip install scipy==1.9.3
!pip install numpy==1.25.1

In [2]:
from IPython.display import clear_output
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import matplotlib.pyplot as plt
from plotly.subplots import make_subplots
# pd.set_option("display.float_format","{:.2f}".format)
pio.templates["mod"] = go.layout.Template(layout=dict(font=dict(family="Fira Code")))
pio.templates.default = "plotly_dark+mod"
from scipy import stats
import statsmodels.api as sm
from sklearn.model_selection import StratifiedKFold,GridSearchCV,StratifiedShuffleSplit
from sklearn.metrics import confusion_matrix,roc_auc_score,f1_score
from sklearn.preprocessing import StandardScaler,OrdinalEncoder,LabelEncoder,OneHotEncoder,MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator,TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression,LogisticRegressionCV
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier,AdaBoostClassifier,VotingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from zipfile import ZipFile
from glob import glob
import sys
import shutil
import warnings
warnings.filterwarnings(action="ignore")

clear_output()

In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("goyaladi/fraud-detection-dataset")

print("Path to dataset files:", path)

100%|██████████| 82.9k/82.9k [00:00<00:00, 59.8MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/goyaladi/fraud-detection-dataset/versions/3


In [4]:
!sudo ls /root/.cache/kagglehub/datasets/goyaladi/fraud-detection-dataset/versions/3/Data

'Customer Profiles'    'Merchant Information'  'Transaction Data'
'Fraudulent Patterns'  'Transaction Amounts'


In [5]:
all_files = glob('/root/.cache/kagglehub/datasets/goyaladi/fraud-detection-dataset/versions/3/**/*.csv',recursive=True)
all_files

['/root/.cache/kagglehub/datasets/goyaladi/fraud-detection-dataset/versions/3/Data/Transaction Amounts/amount_data.csv',
 '/root/.cache/kagglehub/datasets/goyaladi/fraud-detection-dataset/versions/3/Data/Transaction Amounts/anomaly_scores.csv',
 '/root/.cache/kagglehub/datasets/goyaladi/fraud-detection-dataset/versions/3/Data/Transaction Data/transaction_metadata.csv',
 '/root/.cache/kagglehub/datasets/goyaladi/fraud-detection-dataset/versions/3/Data/Transaction Data/transaction_records.csv',
 '/root/.cache/kagglehub/datasets/goyaladi/fraud-detection-dataset/versions/3/Data/Customer Profiles/customer_data.csv',
 '/root/.cache/kagglehub/datasets/goyaladi/fraud-detection-dataset/versions/3/Data/Customer Profiles/account_activity.csv',
 '/root/.cache/kagglehub/datasets/goyaladi/fraud-detection-dataset/versions/3/Data/Fraudulent Patterns/suspicious_activity.csv',
 '/root/.cache/kagglehub/datasets/goyaladi/fraud-detection-dataset/versions/3/Data/Fraudulent Patterns/fraud_indicators.csv',
 '

In [6]:
account_activity = pd.read_csv(all_files[0])

In [7]:
account_activity.head()

,TransactionID,TransactionAmount
0,1,79.413607
1,2,12.053087
2,3,33.310357
3,4,46.121117
4,5,54.051618


In [6]:
def preprocess(all_files):
    global account_activity,amount_data,anomaly_scores,customer_data,fraud_indicators,merchant_data,suspicious_activity,transaction_category_labels,transaction_metadata,transaction_records
    account_activity = pd.read_csv(all_files[5])
    amount_data = pd.read_csv(all_files[0])
    anomaly_scores = pd.read_csv(all_files[1])
    customer_data = pd.read_csv(all_files[4])
    fraud_indicators = pd.read_csv(all_files[7])
    merchant_data = pd.read_csv(all_files[9])
    suspicious_activity = pd.read_csv(all_files[6])
    transaction_category_labels = pd.read_csv(all_files[8])
    transaction_metadata = pd.read_csv(all_files[2])
    transaction_records = pd.read_csv(all_files[3])
    df = pd.merge(left=account_activity,right=customer_data,left_on="CustomerID",right_on="CustomerID")
    df = pd.merge(left=df,right=transaction_records,left_on="CustomerID",right_on="CustomerID")
    df = pd.merge(left=df,right=suspicious_activity,left_on="CustomerID",right_on="CustomerID")
    df = pd.merge(left=df,right=transaction_metadata,left_on="TransactionID",right_on="TransactionID")
    df = pd.merge(left=df,right=amount_data,left_on="TransactionID",right_on="TransactionID")
    df = pd.merge(left=df,right=fraud_indicators,left_on="TransactionID",right_on="TransactionID")
    df = pd.merge(left=df,right=anomaly_scores,left_on="TransactionID",right_on="TransactionID")
    df = pd.merge(left=df,right=transaction_category_labels,left_on="TransactionID",right_on="TransactionID")
    df = pd.merge(left=df,right=merchant_data,left_on="MerchantID",right_on="MerchantID")
    df.drop(['Name','Address','MerchantName','Location','LastLogin','TransactionID','MerchantID','CustomerID'],axis=1,inplace=True)
    df['Timestamp'] = pd.to_datetime(df['Timestamp'])
    return df

LastLogin Column has been omitted as there are some clashes between Timestamp column and LastLogin<br>
some LastLogin dates pre-dated the Timestamp column values which should be statistically impossible

In [7]:
from ast import AnnAssign
df = preprocess(all_files)

<font size=4>

|Column Name|Type of column|Column Description|
|------|-----|-----|
|Customer ID|Categorical|Value counts will tell us how many transactions they have done|
|Account Balance|Continuous|The Amount of money left in their bank account|
|Age|Continuous|Ages at which they made a transactions|
|Transaction ID|Categorical|Unique IDs given to transactions|
|Transaction Amount|Continuous|Amount of the transactions thats carried out|
|Suspicious Flag|Categorical|Suspicious flag 0 or 1, 0 for not 1 for yes|
|Timestamp|Continuous|Time at which transaction has been carried out|
|Merchant ID|Categorical|Unique ID at the the transaction has been carried out|
|Amount|Continuous|Transaction Amount of the fraudulent activity|
|Fraud Indicator|Categorical|Whether its been flagged Fraud or not|
|Anomaly Score|Continuous|The score given for its potential fraud|
|Category|Categorical|Categories at which they made the transactions|

</font>

In [8]:
time_index = pd.date_range(start=df["Timestamp"].min(),end=df["Timestamp"].max(),freq="H")
display(time_index[0])
time_index[-1]

Timestamp('2022-01-01 00:00:00')

Timestamp('2022-02-11 15:00:00')

In [9]:
temp = df.copy()
temp = temp.set_index('Timestamp').sort_index()
temp["weekday"] = temp.index.day_name()
temp["Hour"] = temp.index.strftime("%H")
temp["Working"] = np.nan
temp.loc[temp.between_time(start_time="9:00:00",end_time="17:00:00").index,"Working"] = 1
temp.fillna(0,inplace=True)
temp["Day"] = np.nan
temp.loc[temp.between_time(start_time="6:00:00",end_time="18:00:00").index,"Day"] = 1
temp.fillna(0,inplace=True)
temp.head()

,AccountBalance,Age,Amount,SuspiciousFlag,TransactionAmount,FraudIndicator,AnomalyScore,Category,weekday,Hour,Working,Day
Timestamp,,,,,,,,,,,,
2022-01-01 00:00:00,2869.689912,50,55.530334,0,79.413607,0,0.686699,Other,Saturday,00,0.0,0.0
2022-01-01 01:00:00,9527.947107,46,12.881180,0,12.053087,0,0.081749,Online,Saturday,01,0.0,0.0
2022-01-01 02:00:00,9288.355525,34,50.176322,0,33.310357,0,0.023857,Travel,Saturday,02,0.0,0.0
2022-01-01 03:00:00,5588.049942,33,41.634001,0,46.121117,0,0.876994,Travel,Saturday,03,0.0,0.0
2022-01-01 04:00:00,7324.785332,18,78.122853,0,54.051618,0,0.034059,Other,Saturday,04,0.0,0.0


In [10]:
temp.columns

Index(['AccountBalance', 'Age', 'Amount', 'SuspiciousFlag',
       'TransactionAmount', 'FraudIndicator', 'AnomalyScore', 'Category',
       'weekday', 'Hour', 'Working', 'Day'],
      dtype='object')

In [11]:
temp.describe().round(2)

,AccountBalance,Age,Amount,SuspiciousFlag,TransactionAmount,FraudIndicator,AnomalyScore,Working,Day
count,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00
mean,5715.46,39.85,55.39,0.02,55.85,0.04,0.49,0.38,0.54
std,2540.52,13.07,25.07,0.16,26.09,0.21,0.29,0.48,0.50
min,1056.30,18.00,10.01,0.00,10.06,0.00,0.00,0.00,0.00
25%,3489.55,29.00,34.50,0.00,33.88,0.00,0.25,0.00,0.00
50%,5753.01,39.00,57.84,0.00,55.96,0.00,0.49,0.00,1.00
75%,7925.71,51.00,75.86,0.00,77.59,0.00,0.74,1.00,1.00
max,9999.78,64.00,99.89,1.00,99.78,1.00,1.00,1.00,1.00


|index|AccountBalance|Age|Amount|SuspiciousFlag|TransactionAmount|FraudIndicator|AnomalyScore|Hour|Working|Day|
|---|---|---|---|---|---|---|---|---|---|---|
|count|1000\.0|1000\.0|1000\.0|1000\.0|1000\.0|1000\.0|1000\.0|1000\.0|1000\.0|1000\.0|
|mean|5715\.46|39\.85|55\.39|0\.02|55\.85|0\.04|0\.49|11\.44|0\.38|0\.54|
|std|2540\.52|13\.07|25\.07|0\.16|26\.09|0\.21|0\.29|6\.91|0\.48|0\.5|
|min|1056\.3|18\.0|10\.01|0\.0|10\.06|0\.0|0\.0|0\.0|0\.0|0\.0|
|25%|3489\.55|29\.0|34\.5|0\.0|33\.88|0\.0|0\.25|5\.0|0\.0|0\.0|
|50%|5753\.01|39\.0|57\.84|0\.0|55\.96|0\.0|0\.49|11\.0|0\.0|1\.0|
|75%|7925\.71|51\.0|75\.86|0\.0|77\.59|0\.0|0\.74|17\.0|1\.0|1\.0|
|max|9999\.78|64\.0|99\.89|1\.0|99\.78|1\.0|1\.0|23\.0|1\.0|1\.0|


# Univariate Tests

## Continuous Variables

### Account Balance

In [30]:
px.histogram(temp,x="AccountBalance",marginal="violin").add_vline(x=temp.AccountBalance.mean(),line=dict(dash="dash",color="#202ff5"),annotation=dict(text=f"mean = {temp.AccountBalance.mean():.2f}",y=0.25,font=dict(color="#ffffff",size=20),align="center"))

In [31]:
stats.shapiro(temp.AccountBalance)

ShapiroResult(statistic=np.float64(0.9544821689843274), pvalue=np.float64(4.636657703971989e-17))

In [33]:
data = sm.qqplot(temp.AccountBalance,line="s").gca().lines
plt.close()
fig = go.Figure()
fig.add_trace(go.Scatter(x=data[0].get_xdata(),y=data[0].get_ydata(),mode="markers",name="Obtained<br>Quantiles"))
fig.add_trace(go.Scatter(x=data[1].get_xdata(),y=data[1].get_ydata(),mode="lines",name="Expected<br>Quantiles"))
fig.update_layout(width=700)

By both tests
- p value ~ 0  


we can conclude that it is not a normal distribution and by the shape of q-qplot we can confirm it is a under-dispersed data and closely resembling uniform distribution data

The distribution is an unknown distribution but is a under-dispersed distribution

### Age

In [83]:
px.histogram(temp,x="Age",marginal="violin").add_vline(x=temp.Age.mean(),line=dict(dash="dash",color="#202ff5"),annotation=dict(text=f"mean = {temp.Age.mean():.2f}",y=0.25,font=dict(color="#ffffff",size=20),align="center"))

In [84]:
stats.shapiro(temp.Age)

ShapiroResult(statistic=np.float64(0.9595718189390523), pvalue=np.float64(5.283281809967244e-16))

In [85]:
stats.normaltest(temp.Age.to_numpy())

NormaltestResult(statistic=np.float64(429.2519607880808), pvalue=np.float64(6.153483519531615e-94))

In [86]:
data = sm.qqplot(temp.Age,line="s").gca().lines
plt.close()
fig = go.Figure()
fig.add_trace(go.Scatter(x=data[0].get_xdata(),y=data[0].get_ydata(),mode="markers",name="Obtained<br>Quantiles"))
fig.add_trace(go.Scatter(x=data[1].get_xdata(),y=data[1].get_ydata(),mode="lines",name="Expected<br>Quantiles"))
fig.update_layout(width=700)

Age is also following a considerable deviation normal distribution

This distribution follows uniform at low significance and also shows normal so it can be a mixture of distributions

### Amount

In [40]:
px.histogram(temp,x="TransactionAmount",marginal="violin").add_vline(x=temp.TransactionAmount.mean(),line=dict(color="#202fff",dash='dash'),annotation=dict(text=f"mean : {temp.TransactionAmount.mean():.2f}",y=0.5))

In [41]:
data = sm.qqplot(temp.TransactionAmount.to_numpy(),line="s").gca().lines
plt.close()
fig = go.Figure()
fig.add_trace(go.Scatter(x=data[0].get_xdata(),y=data[0].get_ydata(),mode="markers",name="Observred<br>Quantiles"))
fig.add_trace(go.Scatter(x=data[1].get_xdata(),y=data[1].get_ydata(),mode="lines",name="Expeceted<br>Quantiles"))
fig.update_layout(width=700)
fig.show()

This is definitely a uniform distribution

## New

In [107]:
import plotly.express as px

# Separate fraud and non-fraud data
fraud_df = temp[temp["FraudIndicator"] == 1]
nonfraud_df = temp[temp["FraudIndicator"] == 0]

# Create base histogram for non-fraud
fig = px.histogram(
    nonfraud_df, x="TransactionAmount", nbins=50, opacity=0.6,
    histnorm="probability density", color_discrete_sequence=["blue"],
    labels={"TransactionAmount": "Transaction Amount"}
)

# Add fraud histogram on top
fig.add_trace(
    px.histogram(
        fraud_df, x="TransactionAmount", nbins=50, opacity=0.6,
        histnorm="probability density", color_discrete_sequence=["red"]
    ).data[0]
)

# Update layout and legend
fig.update_layout(
    title="1️⃣ Transaction Amount Distribution (Fraud vs. Non-Fraud)",
    xaxis_title="Transaction Amount",
    yaxis_title="Density",
    barmode="overlay",
    legend=dict(title="FraudIndicator"),
)

# Rename traces manually
fig.data[0].name = "Non-Fraud (0)"
fig.data[1].name = "Fraud (1)"

fig.show()


In [16]:
import plotly.graph_objects as go

# Ensure Hour is numeric
temp["Hour"] = temp["Hour"].astype(int)

# Get fraud counts per hour
fraud_by_hour = temp[temp["FraudIndicator"] == 1]["Hour"].value_counts().sort_index()
fraud_by_hour_df = fraud_by_hour.reset_index()
fraud_by_hour_df.columns = ["Hour", "Fraud Count"]

# Create line plot
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=fraud_by_hour_df["Hour"],
    y=fraud_by_hour_df["Fraud Count"],
    mode="lines+markers",
    line=dict(color="red", width=3),
    marker=dict(size=8),
    name="Fraud Count"
))

fig.update_layout(
    title="2️⃣ Time of Transaction vs. Fraud Occurrence",
    xaxis_title="Hour of Day",
    yaxis_title="Number of Fraudulent Transactions",
    xaxis=dict(dtick=1),
    template="plotly_dark"  # Optional: for dark mode style
)

fig.show()


In [17]:
import pandas as pd
import plotly.express as px

# Step 1: Group by Category and calculate fraud rate
fraud_rate_df = (
    temp.groupby("Category")["FraudIndicator"]
    .agg(["count", "sum"])  # count = total transactions, sum = frauds
    .rename(columns={"count": "Total", "sum": "Fraudulent"})
)

fraud_rate_df["Fraud Rate (%)"] = (fraud_rate_df["Fraudulent"] / fraud_rate_df["Total"]) * 100
fraud_rate_df = fraud_rate_df.reset_index()

# Step 2: Plot
fig = px.bar(
    fraud_rate_df,
    x="Category",
    y="Fraud Rate (%)",
    text="Fraud Rate (%)",
    title="4️⃣ Fraud Rate by Category",
    labels={"Category": "Transaction Category", "Fraud Rate (%)": "Fraud %"},
    color="Fraud Rate (%)",
    color_continuous_scale="Reds"
)

fig.update_layout(xaxis_title="Category", yaxis_title="% Fraudulent Transactions")
fig.show()


### Transaction Amount

In [18]:
px.histogram(temp,x="TransactionAmount",marginal="violin").add_vline(x=temp.TransactionAmount.mean(),line=dict(color="#202fff",dash='dash'),annotation=dict(text=f"mean : {temp.TransactionAmount.mean():.2f}",y=0.5))

In [44]:
stats.shapiro(temp.TransactionAmount)

ShapiroResult(statistic=np.float64(0.9560227349605414), pvalue=np.float64(9.480801041048785e-17))

In [45]:
stats.normaltest(temp.TransactionAmount)

NormaltestResult(statistic=np.float64(563.7952795850348), pvalue=np.float64(3.7446443496735614e-123))

This distribution closely follows a uniform distribution

## Categorical Variables

In [20]:
temp.head(2)

,AccountBalance,Age,Amount,SuspiciousFlag,TransactionAmount,FraudIndicator,AnomalyScore,Category,weekday,Hour,Working,Day
Timestamp,,,,,,,,,,,,
2022-01-01 00:00:00,2869.689912,50,55.530334,0,79.413607,0,0.686699,Other,Saturday,0,0.0,0.0
2022-01-01 01:00:00,9527.947107,46,12.881180,0,12.053087,0,0.081749,Online,Saturday,1,0.0,0.0


In [19]:
px.histogram(temp,x="SuspiciousFlag",histfunc="count",title="Suspicious Flag",color="SuspiciousFlag",color_discrete_map={0:"#636efa",1:"red"})

In [49]:
sm.stats.proportions_ztest(count=temp.SuspiciousFlag.value_counts().to_numpy(),nobs=temp.shape[0])

(np.float64(42.485291572496), np.float64(0.0))

In [93]:
temp["FraudIndicator"].value_counts()

,count
FraudIndicator,
0,955
1,45


Suspicous Flag proportions are statsitically significant as p-value is 0 from one-proportions z-test

In [21]:
px.histogram(temp,x="FraudIndicator",histfunc="count",title="Fraud Indicator",color="FraudIndicator",color_discrete_map={0:"#636efa",1:"red"})

In [22]:
sm.stats.proportions_ztest(temp.FraudIndicator.value_counts(),nobs=temp.shape[0])

(np.float64(40.69643719049617), np.float64(0.0))

In [23]:
temp.Category.value_counts()

,count
Category,
Other,210
Food,204
Travel,198
Online,196
Retail,192


In [53]:
fig = make_subplots(cols=2,specs=[[{"type":"xy"},{"type":"domain"}]])
fig.add_trace(go.Bar(x=temp.Category.value_counts().index,y=temp.Category.value_counts(),marker=dict(color=px.colors.qualitative.Plotly)),row=1,col=1)
fig.add_trace(go.Pie(values=temp.Category.value_counts().to_numpy(),labels=temp.Category.value_counts().index,textinfo="label+percent+value",marker=dict(colors=px.colors.qualitative.Plotly),sort=False),row=1,col=2)
fig.update_layout(showlegend=False,title="Category")
fig.update_yaxes(title="count")
fig.update_xaxes(title="category")
fig.show()

In [54]:
print("p-value  :",stats.chi2_contingency([temp.Category.value_counts(),np.full_like(temp.Category.value_counts(),1000/temp.Category.nunique())])[1])

p-value  : 0.9737704658381907


They are equal in proportion

In [55]:
px.histogram(temp,x="weekday",color="weekday",title="Weekday").update_layout(showlegend=False)

In [56]:
print("p-value  :",stats.chi2_contingency([temp.weekday.value_counts(),np.full_like(temp.weekday.value_counts(),1000/temp.weekday.nunique())])[1])

p-value  : 0.9998556130268133


In [57]:
px.histogram(temp,x="Hour",color="Hour").update_layout(showlegend=False)

In [58]:
print("p-value  :",stats.chi2_contingency([temp.Hour.value_counts(),np.full_like(temp.Hour.value_counts(),1000/temp.Hour.nunique())])[1])

p-value  : 1.0


Fraud Indicator proportions are statsitically significant as p-value is 0 from one-proportions z-test

# Double Variable Tests

## Countinous

### Between AccountBalance and TransactionAmount

In [59]:
cont_cols = ["TransactionAmount","Amount","AccountBalance","AnomalyScore","Age"]

In [60]:
corr_arr = temp[cont_cols].corr()
np.fill_diagonal(corr_arr.to_numpy(),0)
fig = px.imshow(corr_arr,text_auto=".2f",color_continuous_scale="GnBu_r",title="Correlation between all the continuous Columns")
fig.add_annotation(text="same columns values are made zero<br>to understand <br>the relative dependence",x=1.05,y=0,xanchor="left",yanchor="bottom",xref="x domain",yref="y domain",showarrow=False)
fig.show()

## Categorical Variables

### Contingency test between FraudIndicator and the rest of the categorical columns

In [65]:
temp.columns

Index(['AccountBalance', 'Age', 'Amount', 'SuspiciousFlag',
       'TransactionAmount', 'FraudIndicator', 'AnomalyScore', 'Category',
       'weekday', 'Hour', 'Working', 'Day'],
      dtype='object')

In [66]:
def givemat(col):
    return pd.crosstab(index=temp.FraudIndicator,columns=temp[col])

In [67]:
print(f'''p-value for Working :{sm.stats.mcnemar(givemat("Working").to_numpy()).pvalue}''')
print(f'''p-value for Hour :{stats.chi2_contingency(givemat("Hour").to_numpy())[1]}''')
print(f'''p-value for Day :{sm.stats.mcnemar(givemat("Day").to_numpy()).pvalue}''')
print(f'''p-value for Category :{stats.chi2_contingency(givemat("Category").to_numpy())[1]}''')

p-value for Working :3.687264509158562e-71
p-value for Hour :0.7958505114310169
p-value for Day :2.4846719053465266e-122
p-value for Category :0.9577050458132295


As we can see Working and Day are having dependency as they can be considered paired because of how the time dilation

In [87]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=temp.query("FraudIndicator == 1 and Working == 0").index,
    y=temp.query("FraudIndicator == 1 and Working == 0").Amount,
    marker=dict(size=temp.query("FraudIndicator == 1 and Working == 0").AnomalyScore,sizeref=0.03,sizemin=4,color="blue",line_color="white"),
    mode="markers",
    name="Fraudulent at \n<br>Non-Working Hours\n<br>5 PM to 8 AM"
    ))
fig.add_trace(go.Scatter(
    x=temp.query("FraudIndicator == 1 and Working == 1").index,
    y=temp.query("FraudIndicator == 1 and Working == 1").Amount,
    marker=dict(size=temp.query("FraudIndicator == 1 and Working == 1").AnomalyScore,sizeref=0.03,sizemin=4,color="red",line_color="white"),
    mode="markers",
    name="Fraudulent at \n<br>Working Hours\n<br>8 AM to 5 PM"
    ))
fig.add_trace(go.Scatter(
    x=temp.query("FraudIndicator == 0 and Working == 0").index,
    y=temp.query("FraudIndicator == 0 and Working == 0").Amount,
    marker=dict(size=temp.query("FraudIndicator == 0 and Working == 0").AnomalyScore,sizeref=0.03,sizemin=4,color="green",opacity=0.3),
    mode="markers",
    name="Non-Fraudulent at \n<br>Non-Working Hours\n<br>5 PM to 8 AM",
    visible="legendonly"
))
fig.add_trace(go.Scatter(
    x=temp.query("FraudIndicator == 0 and Working == 1").index,
    y=temp.query("FraudIndicator == 0 and Working == 1").Amount,
    marker=dict(size=temp.query("FraudIndicator == 0 and Working == 1").AnomalyScore,sizeref=0.03,sizemin=4,color="yellow",opacity=0.3),
    mode="markers",
    name="Non-Fraudulent at \n<br>Working Hours\n<br>8 AM to 5 PM",
    visible="legendonly"
))
fig.add_annotation(text="*Toggle normal to see <br>the non-fraudulent transactions<br> that took place",x=1.25,y=-0.15,xanchor="right",yanchor="bottom",showarrow=False,xref="x domain",yref="y domain")
fig.add_annotation(text="*size indicates AnomalyScore",x=1.25,y=0,xanchor="right",yanchor="bottom",showarrow=False,xref="x domain",yref="y domain")
fig.update_layout(title=dict(text="Fraudulent Activity by Time from January 1,2022 to February 11,2022"),legend=dict(itemsizing="constant"))
fig.update_xaxes(title=dict(text="Time"))
fig.update_yaxes(title=dict(text="Amount"))
fig.show()

Most fraudulent activity occurs during non-working hours:

The plot is clearly dominated by blue bubbles.

Suggests fraudsters may target off-hours to avoid detection.

Higher anomaly scores also tend to appear off-hours:

Larger blue bubbles are common, meaning your detection model assigns higher AnomalyScores to these cases.

Red (working hour) fraud does exist:

But it’s less frequent and usually smaller in size.

##### There are quite a bit of Fraudulent Transactions that took place at Non-Working hours

Fraudulent Transactions percentage at Non working Hours on total transactions :

In [88]:
temp.query("FraudIndicator == 1 and Working == 0").shape[0]/temp.query("FraudIndicator == 1").shape[0]

0.7555555555555555

Fraudulent Transactions percentage at Working Hours on total transactions :

In [89]:
temp.query("FraudIndicator == 1 and Working == 1").shape[0]/temp.query("FraudIndicator == 1").shape[0]

0.24444444444444444

## Desicion Tree

### No Grid Search

In [12]:
# Features and target
X = temp[["Amount", "AnomalyScore", "Working", "Day"]]
y = temp["FraudIndicator"]

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

# bScale features (optional for decision trees, but consistent if done before)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Initialize and train Decision Tree (no grid search)
clf = DecisionTreeClassifier(
    criterion="gini",       # or 'entropy'
    max_depth=5,            # You can adjust this manually
    min_samples_split=2,
    class_weight="balanced",  # Handle imbalance
    random_state=42
)
clf.fit(X_train_scaled, y_train)

# Evaluate
y_pred = clf.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


Accuracy: 0.88
Classification Report:
               precision    recall  f1-score   support

           0       0.96      0.92      0.94       287
           1       0.04      0.08      0.05        13

    accuracy                           0.88       300
   macro avg       0.50      0.50      0.49       300
weighted avg       0.92      0.88      0.90       300

Confusion Matrix:
 [[263  24]
 [ 12   1]]


### Using Grid Search

In [25]:
# Define features and target
X = temp[["Amount", "AnomalyScore", "Working", "Day"]]
y = temp["FraudIndicator"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Scale the features (especially useful for continuous variables)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Set up Decision Tree and Grid Search
dtree = DecisionTreeClassifier(random_state=42, class_weight='balanced')

param_grid = {
    'max_depth': [3, 5, 10, None],
    'min_samples_split': [2, 5, 10],
    'criterion': ['gini', 'entropy']
}

grid = GridSearchCV(dtree, param_grid, cv=5, scoring='f1', n_jobs=-1)
grid.fit(X_train_scaled, y_train)

# Best model
best_model = grid.best_estimator_
print("Best parameters:", grid.best_params_)

# Make predictions
y_pred = best_model.predict(X_test_scaled)

# Evaluate
print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))


Best parameters: {'criterion': 'gini', 'max_depth': None, 'min_samples_split': 5}

Accuracy: 0.9

Classification Report:
               precision    recall  f1-score   support

           0       0.96      0.94      0.95       287
           1       0.05      0.08      0.06        13

    accuracy                           0.90       300
   macro avg       0.50      0.51      0.50       300
weighted avg       0.92      0.90      0.91       300


Confusion Matrix:
 [[269  18]
 [ 12   1]]


In [26]:
# 1. Create DataFrame from importances and features
feature_importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": best_model.feature_importances_
}).sort_values(by="Importance", ascending=True)

# 2. Plot with Plotly
fig = px.bar(
    feature_importance_df,
    x="Importance",
    y="Feature",
    orientation="h",
    title="Feature Importance - Decision Tree",
    color="Importance",
    color_continuous_scale="Greens",
    labels={"Importance": "Feature Importance", "Feature": "Features"}
)

fig.update_layout(height=500)
fig.show()


## Random Forest

### No Grid Search

In [13]:
# Features and target
X = temp[["Amount", "AnomalyScore", "Working", "Day"]]
y = temp["FraudIndicator"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

# Optional: Scale (not needed for trees, but for consistency if others do)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Random Forest model (no GridSearch)
rf = RandomForestClassifier(
    n_estimators=100,          # Number of trees
    max_depth=7,               # Set manually, or use None for full depth
    min_samples_split=2,       # Minimum samples to split a node
    min_samples_leaf=1,        # Minimum samples per leaf
    criterion='gini',          # or 'entropy'
    class_weight='balanced',   # Important for fraud (imbalanced)
    random_state=42
)

# Train the model
rf.fit(X_train_scaled, y_train)

# Predict and evaluate
y_pred = rf.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.9533333333333334
Classification Report:
               precision    recall  f1-score   support

           0       0.96      1.00      0.98       287
           1       0.00      0.00      0.00        13

    accuracy                           0.95       300
   macro avg       0.48      0.50      0.49       300
weighted avg       0.92      0.95      0.93       300

Confusion Matrix:
 [[286   1]
 [ 13   0]]


### Using Grid Search

In [29]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Select your features and target
X = temp[["Amount", "AnomalyScore", "Working", "Day"]]  # add more if relevant
y = temp["FraudIndicator"]


# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

# Feature scaling (optional for tree-based models, but good practice)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Initialize Random Forest with class_weight for imbalance
rf = RandomForestClassifier(class_weight="balanced", random_state=42)

# Define grid of hyperparameters
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [5, 10, None],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "criterion": ["gini", "entropy"]
}

# GridSearchCV
grid = GridSearchCV(rf, param_grid, scoring="f1", cv=5, n_jobs=-1)
grid.fit(X_train_scaled, y_train)

# Get best model
best_rf = grid.best_estimator_
print("Best parameters:", grid.best_params_)

# Make predictions
y_pred = best_rf.predict(X_test_scaled)

# Evaluate the model
print("\n Accuracy:", accuracy_score(y_test, y_pred))
print("\n Classification Report:\n", classification_report(y_test, y_pred))
print("\n Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

Best parameters: {'criterion': 'gini', 'max_depth': 5, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}

 Accuracy: 0.9266666666666666

 Classification Report:
               precision    recall  f1-score   support

           0       0.96      0.97      0.96       287
           1       0.00      0.00      0.00        13

    accuracy                           0.93       300
   macro avg       0.48      0.48      0.48       300
weighted avg       0.91      0.93      0.92       300


 Confusion Matrix:
 [[278   9]
 [ 13   0]]


In [97]:
import seaborn as sns

In [32]:
features= ["Amount", "AnomalyScore", "Working", "Day"]

# Assuming 'features' is a list of feature names
feature_importance_df = pd.DataFrame({
    "Feature": features,
    "Importance": best_rf.feature_importances_
}).sort_values(by="Importance", ascending=True)  # Sort for better display

# Plot
fig = px.bar(
    feature_importance_df,
    x="Importance",
    y="Feature",
    orientation="h",  # horizontal bars
    title="Feature Importance - Random Forest",
    labels={"Importance": "Feature Importance", "Feature": "Features"},
    color="Importance",
    color_continuous_scale="Blues"
)

fig.update_layout(yaxis=dict(tickfont=dict(size=12)), height=500)
fig.show()

## XGBoost

### No Grid Search

In [33]:
# Define features and target
X = temp[["Amount", "AnomalyScore", "Working", "Day"]]
y = temp["FraudIndicator"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

# (Optional) Scaling — not required for XGBoost but keeps consistency
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Calculate scale_pos_weight to address class imbalance
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

# Initialize and train XGBoost
xgb = XGBClassifier(
    n_estimators=50,
    max_depth=5,
    learning_rate=0.1,
    subsample=1,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='logloss',
    scale_pos_weight=scale_pos_weight,
    random_state=42
)
xgb.fit(X_train_scaled, y_train)

# Predict and evaluate
y_pred = xgb.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.8766666666666667
Classification Report:
               precision    recall  f1-score   support

           0       0.96      0.91      0.93       287
           1       0.04      0.08      0.05        13

    accuracy                           0.88       300
   macro avg       0.50      0.49      0.49       300
weighted avg       0.92      0.88      0.90       300

Confusion Matrix:
 [[262  25]
 [ 12   1]]


### Using Grid Search

In [34]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Select features and target
X = temp[["Amount", "AnomalyScore", "Working", "Day"]]  # Add more features as needed
y = temp["FraudIndicator"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

# Optional scaling (XGBoost doesn't need it, but can help for stability in some setups)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Initialize XGBoost with class_weight equivalent
# Use scale_pos_weight to handle imbalance: ratio of negatives to positives
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss', scale_pos_weight=scale_pos_weight, random_state=42)

# Grid of hyperparameters
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.05, 0.1],
    "subsample": [0.8, 1],
    "colsample_bytree": [0.8, 1]
}

# Grid search
grid = GridSearchCV(xgb, param_grid, scoring="f1", cv=5, n_jobs=-1)
grid.fit(X_train_scaled, y_train)

# Best model
best_xgb = grid.best_estimator_
print("Best parameters:", grid.best_params_)

# Predictions
y_pred = best_xgb.predict(X_test_scaled)

# Evaluation
print("\n Accuracy:", accuracy_score(y_test, y_pred))
print("\n Classification Report:\n", classification_report(y_test, y_pred))
print("\n Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


Best parameters: {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 5, 'n_estimators': 100, 'subsample': 1}

 Accuracy: 0.8666666666666667

 Classification Report:
               precision    recall  f1-score   support

           0       0.96      0.90      0.93       287
           1       0.03      0.08      0.05        13

    accuracy                           0.87       300
   macro avg       0.50      0.49      0.49       300
weighted avg       0.92      0.87      0.89       300


 Confusion Matrix:
 [[259  28]
 [ 12   1]]


In [35]:
import plotly.express as px
import pandas as pd

# Create DataFrame of feature importances
feature_importance_df = pd.DataFrame({
    "Feature": features,
    "Importance": best_xgb.feature_importances_
}).sort_values(by="Importance", ascending=True)

# Plot using Plotly
fig = px.bar(
    feature_importance_df,
    x="Importance",
    y="Feature",
    orientation="h",
    title="Feature Importance - XGBoost",
    color="Importance",
    color_continuous_scale="OrRd",
    labels={"Importance": "Feature Importance", "Feature": "Features"}
)

fig.update_layout(height=500)
fig.show()


In [116]:
import plotly.express as px
import pandas as pd

# Data
data = {
    "Model": ["Decision Tree", "Random Forest", "XGBoost"],
    "F1 Class 1": [0.06, 0.00, 0.05],
    "Overall Accuracy": [0.90, 0.93, 0.87]
}

df = pd.DataFrame(data)

# Melt to long format
df_melted = df.melt(id_vars="Model", var_name="Metric", value_name="Score")

# Plot
fig = px.scatter(
    df_melted,
    x="Model",
    y="Score",
    color="Metric",
    symbol="Metric",
    size="Score",
    text="Score",
    title="📈 Comparison of Key Metrics Across Models",
)

fig.update_traces(textposition="top center")
fig.update_layout(yaxis_title="Score", xaxis_title="Model", legend_title="Metric")
fig.show()
